# Team Operations : Reset, Stop, Resume and Abort

In [28]:
import asyncio
from autogen_ext.models.openai import OpenAIChatCompletionClient
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv('OPENAI_API_KEY')
model_client = OpenAIChatCompletionClient(
    model="openai/gpt-oss-20b",
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1",
    model_info={
        "family": "gpt-4o",
        "vision": True,
        "function_calling": True,
        "json_output": True,
        "structured_output": True,
    },
)

In [29]:
from autogen_agentchat.agents import AssistantAgent
add_1_agent_first = AssistantAgent(
    name = 'add_1_agent_first',
    model_client=model_client,
    system_message="Add 1 to the number, first number is 0. Give result as output"
)

add_1_agent_second = AssistantAgent(
    name = 'add_1_agent_second',
    model_client=model_client,
    system_message="Add 1 to the number. Give result as output."
)
 
add_1_agent_third = AssistantAgent(
    name = 'add_1_agent_third',
    model_client=model_client,
    system_message="Add 1 to the number. Give result as output."
)


In [30]:
from autogen_agentchat.teams import RoundRobinGroupChat

team = RoundRobinGroupChat(
    [add_1_agent_first, add_1_agent_second, add_1_agent_third],
    max_turns=3
)

In [31]:
from autogen_agentchat.ui import Console

await Console(team.run_stream())

---------- TextMessage (add_1_agent_first) ----------
1
---------- TextMessage (add_1_agent_second) ----------
2
---------- TextMessage (add_1_agent_third) ----------
2
3


TaskResult(messages=[TextMessage(source='add_1_agent_first', models_usage=RequestUsage(prompt_tokens=87, completion_tokens=65), metadata={}, content='1', type='TextMessage'), TextMessage(source='add_1_agent_second', models_usage=RequestUsage(prompt_tokens=92, completion_tokens=56), metadata={}, content='2', type='TextMessage'), TextMessage(source='add_1_agent_third', models_usage=RequestUsage(prompt_tokens=102, completion_tokens=141), metadata={}, content='2\n3', type='TextMessage')], stop_reason='Maximum number of turns 3 reached.')

# Resuming a Team
Teams are stateful and maintains the conversation history and context after each run, unless you reset the team.


We can resume a team to continue from where it left off by calling the run() or run_stream() method without a **new task**


In [32]:
await Console(team.run_stream())

---------- TextMessage (add_1_agent_first) ----------
3
---------- TextMessage (add_1_agent_second) ----------
4
---------- TextMessage (add_1_agent_third) ----------
4
5


TaskResult(messages=[TextMessage(source='add_1_agent_first', models_usage=RequestUsage(prompt_tokens=117, completion_tokens=681), metadata={}, content='3', type='TextMessage'), TextMessage(source='add_1_agent_second', models_usage=RequestUsage(prompt_tokens=122, completion_tokens=738), metadata={}, content='4', type='TextMessage'), TextMessage(source='add_1_agent_third', models_usage=RequestUsage(prompt_tokens=131, completion_tokens=481), metadata={}, content='4\n5', type='TextMessage')], stop_reason='Maximum number of turns 3 reached.')

In [34]:
await Console(team.run_stream())

---------- TextMessage (add_1_agent_first) ----------
Error: No number provided.
---------- TextMessage (add_1_agent_second) ----------
Error: No number provided.
---------- TextMessage (add_1_agent_third) ----------
Error: No number provided.


TaskResult(messages=[TextMessage(source='add_1_agent_first', models_usage=RequestUsage(prompt_tokens=179, completion_tokens=413), metadata={}, content='Error: No number provided.', type='TextMessage'), TextMessage(source='add_1_agent_second', models_usage=RequestUsage(prompt_tokens=189, completion_tokens=168), metadata={}, content='Error: No number provided.', type='TextMessage'), TextMessage(source='add_1_agent_third', models_usage=RequestUsage(prompt_tokens=201, completion_tokens=327), metadata={}, content='Error: No number provided.', type='TextMessage')], stop_reason='Maximum number of turns 3 reached.')

# team resumed from where it left off in the output above, and the first message is from the next agent after the last agent that spoke before the team stopped.

In [35]:
await Console(team.run_stream(task = 'What was the largest number you got in the result?'))

---------- TextMessage (user) ----------
What was the largest number you got in the result?
---------- TextMessage (add_1_agent_first) ----------
The largest number I produced was **5**.
---------- TextMessage (add_1_agent_second) ----------
The largest number you produced was **5**.
---------- TextMessage (add_1_agent_third) ----------
5


TaskResult(messages=[TextMessage(source='user', models_usage=None, metadata={}, content='What was the largest number you got in the result?', type='TextMessage'), TextMessage(source='add_1_agent_first', models_usage=RequestUsage(prompt_tokens=238, completion_tokens=456), metadata={}, content='The largest number I produced was **5**.', type='TextMessage'), TextMessage(source='add_1_agent_second', models_usage=RequestUsage(prompt_tokens=252, completion_tokens=71), metadata={}, content='The largest number you produced was **5**.', type='TextMessage'), TextMessage(source='add_1_agent_third', models_usage=RequestUsage(prompt_tokens=267, completion_tokens=761), metadata={}, content='5', type='TextMessage')], stop_reason='Maximum number of turns 3 reached.')

# Reset our Team

In [39]:
await team.reset() # on_reset() on all agents

In [40]:
await Console(team.run_stream())

---------- add_1_agent_first ----------
1
---------- add_1_agent_second ----------
2
---------- add_1_agent_third ----------
3


TaskResult(messages=[TextMessage(source='add_1_agent_first', models_usage=RequestUsage(prompt_tokens=24, completion_tokens=2), metadata={}, content='1', type='TextMessage'), TextMessage(source='add_1_agent_second', models_usage=RequestUsage(prompt_tokens=29, completion_tokens=2), metadata={}, content='2', type='TextMessage'), TextMessage(source='add_1_agent_third', models_usage=RequestUsage(prompt_tokens=39, completion_tokens=2), metadata={}, content='3', type='TextMessage')], stop_reason='Maximum number of turns 3 reached.')

## Covered in Future Videos in the Module

# Aborting a Team

Different from stopping a team, aborting a team will immediately stop the team and raise a CancelledError exception.

In [36]:
from autogen_core import CancellationToken

cancellation_token = CancellationToken()

run2 = asyncio.create_task(
    Console(team.run_stream(task = 'Give a short Story about a lion atmost 40 words',cancellation_token=cancellation_token))
)

await asyncio.sleep(2)
cancellation_token.cancel()

try:
    result = await run2
except asyncio.CancelledError():
    print("Task Was Cancelled")

---------- TextMessage (user) ----------
Give a short Story about a lion atmost 40 words


---------- TextMessage (add_1_agent_first) ----------
In the golden savanna, Leo the lion roared, not for battle but to sing to the stars. His voice echoed, calming the restless wind, and the jungle fell into peaceful sleep.
---------- TextMessage (add_1_agent_second) ----------
In twilight savanna, a lone lion named Arin surveyed the horizon. With each silent step, he guarded the night, his mane glinting like moonlit sand. A gentle purr whispered to the wind, promising dawn’s hopeful roar.


Error processing publish message for add_1_agent_third_b8743300-e5b8-4307-9025-625fd05be987/b8743300-e5b8-4307-9025-625fd05be987
Traceback (most recent call last):
  File "/Users/vishalsharma/AI/Autogen/venv/lib/python3.14/site-packages/openai/_base_client.py", line 1696, in request
    response.raise_for_status()
    ~~~~~~~~~~~~~~~~~~~~~~~~~^^
  File "/Users/vishalsharma/AI/Autogen/venv/lib/python3.14/site-packages/httpx/_models.py", line 829, in raise_for_status
    raise HTTPStatusError(message, request=request, response=self)
httpx.HTTPStatusError: Client error '429 Too Many Requests' for url 'https://api.groq.com/openai/v1/chat/completions'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/429

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/Users/vishalsharma/AI/Autogen/venv/lib/python3.14/site-packages/autogen_core/_single_threaded_agent_runtime.py", line 604, in _on_message
    

TypeError: catching classes that do not inherit from BaseException is not allowed